In [1]:

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Locate project directory
# --------------------------------------------------

PROJECT_DIR = Path.cwd()

if PROJECT_DIR.name.lower() == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

WEATHER_FILE = (
    PROJECT_DIR
    / "data"
    / "processed"
    / "weather_operational_combined_2006_2026.csv"
)

print("=" * 65)
print("ENVIRONMENTAL ANOMALY DETECTION")
print("STEP 1: WEATHER DATA INSPECTION")
print("=" * 65)

# --------------------------------------------------
# 2. Check file availability
# --------------------------------------------------

if not WEATHER_FILE.exists():
    raise FileNotFoundError(
        f"Weather dataset not found:\n{WEATHER_FILE}"
    )

print("\nDataset found:")
print(WEATHER_FILE)

# --------------------------------------------------
# 3. Load dataset
# --------------------------------------------------

weather_df = pd.read_csv(WEATHER_FILE)

print("\nDATASET DIMENSIONS")
print("-" * 65)
print(f"Rows: {weather_df.shape[0]:,}")
print(f"Columns: {weather_df.shape[1]}")

# --------------------------------------------------
# 4. Inspect columns and sample records
# --------------------------------------------------

print("\nDATASET COLUMNS")
print("-" * 65)
print(weather_df.columns.tolist())

print("\nFIRST 5 RECORDS")
print("-" * 65)
print(weather_df.head())

# --------------------------------------------------
# 5. Check date coverage
# --------------------------------------------------

print("\nDATE COVERAGE")
print("-" * 65)

weather_df["date"] = pd.to_datetime(
    weather_df["date"],
    errors="coerce"
)

print("Earliest date:", weather_df["date"].min())
print("Latest date:", weather_df["date"].max())
print("Invalid dates:", weather_df["date"].isna().sum())

# --------------------------------------------------
# 6. Check district coverage
# --------------------------------------------------

print("\nDISTRICT COVERAGE")
print("-" * 65)

print("Number of districts:",
      weather_df["district"].nunique())

print("\nDistrict names:")
print(sorted(weather_df["district"].dropna().unique()))

# --------------------------------------------------
# 7. Check important weather variables
# --------------------------------------------------

weather_variables = [
    "precipitation_sum",
    "temperature_2m_mean",
    "relative_humidity_2m_mean"
]

print("\nWEATHER VARIABLE AVAILABILITY")
print("-" * 65)

for column in weather_variables:
    if column in weather_df.columns:
        print(f"\n{column}")
        print("Missing values:", weather_df[column].isna().sum())
        print("Minimum:", weather_df[column].min())
        print("Maximum:", weather_df[column].max())
    else:
        print(f"\n{column}: NOT FOUND")

# --------------------------------------------------
# 8. Missing-value summary
# --------------------------------------------------

print("\nMISSING VALUE SUMMARY")
print("-" * 65)

missing_summary = (
    weather_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing_summary[missing_summary > 0])

print("\nSTEP 1 INSPECTION COMPLETED.")


ENVIRONMENTAL ANOMALY DETECTION
STEP 1: WEATHER DATA INSPECTION

Dataset found:
c:\Users\Nethmi\Documents\GitHub\RiskWeave\data\processed\weather_operational_combined_2006_2026.csv

DATASET DIMENSIONS
-----------------------------------------------------------------
Rows: 181,550
Columns: 28

DATASET COLUMNS
-----------------------------------------------------------------
['district', 'date', 'precipitation_sum', 'rain_sum', 'precipitation_hours', 'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min', 'relative_humidity_2m_mean', 'relative_humidity_2m_max', 'relative_humidity_2m_min', 'dew_point_2m_mean', 'wind_speed_10m_mean', 'wind_speed_10m_max', 'wind_direction_10m_dominant', 'cloud_cover_mean', 'shortwave_radiation_sum', 'et0_fao_evapotranspiration', 'requested_latitude', 'requested_longitude', 'era5_grid_latitude', 'era5_grid_longitude', 'era5_grid_elevation_m', 'model', 'timezone', 'data_source', 'hourly_observation_count', 'retrieved_at']

FIRST 5 RECORDS
---------

In [2]:

# ============================================================
# STEP 2: DATA QUALITY AND DAILY CONTINUITY CHECKS
# ============================================================

print("=" * 65)
print("ENVIRONMENTAL ANOMALY DETECTION")
print("STEP 2: DATA QUALITY AND CONTINUITY")
print("=" * 65)

# ------------------------------------------------------------
# 1. Required columns
# ------------------------------------------------------------

required_columns = [
    "district",
    "date",
    "precipitation_sum",
    "temperature_2m_mean",
    "relative_humidity_2m_mean"
]

missing_columns = [
    col for col in required_columns
    if col not in weather_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns missing: {missing_columns}"
    )

analysis_df = weather_df[required_columns].copy()

analysis_df["date"] = pd.to_datetime(
    analysis_df["date"]
)

# ------------------------------------------------------------
# 2. Duplicate district-date records
# ------------------------------------------------------------

print("\n1. DUPLICATE RECORD ANALYSIS")
print("-" * 65)

duplicate_count = analysis_df.duplicated(
    subset=["district", "date"]
).sum()

print("Duplicate district-date records:", duplicate_count)

# ------------------------------------------------------------
# 3. Missing values in selected variables
# ------------------------------------------------------------

print("\n2. SELECTED VARIABLE MISSING VALUES")
print("-" * 65)

print(
    analysis_df[
        required_columns[2:]
    ].isnull().sum()
)

# ------------------------------------------------------------
# 4. District-level record counts
# ------------------------------------------------------------

print("\n3. DISTRICT RECORD COUNTS")
print("-" * 65)

district_counts = (
    analysis_df.groupby("district")
    .agg(
        total_records=("date", "count"),
        first_date=("date", "min"),
        last_date=("date", "max"),
        unique_dates=("date", "nunique")
    )
    .sort_index()
)

print(district_counts.to_string())

# ------------------------------------------------------------
# 5. Check missing calendar days per district
# ------------------------------------------------------------

print("\n4. DAILY DATE CONTINUITY")
print("-" * 65)

continuity_results = []

for district, group in analysis_df.groupby("district"):

    dates = (
        group["date"]
        .dropna()
        .drop_duplicates()
        .sort_values()
    )

    expected_dates = pd.date_range(
        start=dates.min(),
        end=dates.max(),
        freq="D"
    )

    missing_dates = expected_dates.difference(dates)

    continuity_results.append({
        "district": district,
        "expected_days": len(expected_dates),
        "available_days": len(dates),
        "missing_days": len(missing_dates)
    })

continuity_df = pd.DataFrame(continuity_results)

print(continuity_df.to_string(index=False))

# ------------------------------------------------------------
# 6. Physical range checks
# ------------------------------------------------------------

print("\n5. WEATHER VALUE RANGE CHECKS")
print("-" * 65)

range_checks = {
    "precipitation_sum": (
        analysis_df["precipitation_sum"] < 0
    ),
    "relative_humidity_2m_mean": (
        (analysis_df["relative_humidity_2m_mean"] < 0) |
        (analysis_df["relative_humidity_2m_mean"] > 100)
    ),
    "temperature_2m_mean": (
        analysis_df["temperature_2m_mean"].isna()
    )
}

for variable, condition in range_checks.items():
    print(f"{variable}: {condition.sum()} flagged records")

# ------------------------------------------------------------
# 7. Summary
# ------------------------------------------------------------

print("\n6. QUALITY CHECK SUMMARY")
print("-" * 65)

print("Total records:", len(analysis_df))
print("Duplicate records:", duplicate_count)
print(
    "Total missing calendar days:",
    continuity_df["missing_days"].sum()
)

print("\nSTEP 2 COMPLETED.")


ENVIRONMENTAL ANOMALY DETECTION
STEP 2: DATA QUALITY AND CONTINUITY

1. DUPLICATE RECORD ANALYSIS
-----------------------------------------------------------------
Duplicate district-date records: 0

2. SELECTED VARIABLE MISSING VALUES
-----------------------------------------------------------------
precipitation_sum            0
temperature_2m_mean          0
relative_humidity_2m_mean    0
dtype: int64

3. DISTRICT RECORD COUNTS
-----------------------------------------------------------------
              total_records first_date  last_date  unique_dates
district                                                       
Ampara                 7262 2006-12-23 2026-11-09          7262
Anuradhapura           7262 2006-12-23 2026-11-09          7262
Badulla                7262 2006-12-23 2026-11-09          7262
Batticaloa             7262 2006-12-23 2026-11-09          7262
Colombo                7262 2006-12-23 2026-11-09          7262
Galle                  7262 2006-12-23 2026-11-09  